# Medical Benchmark — Non-RAG Phases

Evaluates non-RAG phases (no external retriever) on MedQA and PubMedQA.

| Cell | Phase | Model | Dataset | Context |
|------|-------|-------|---------|--------|
| 5 | **Phase 1** | RAG-SFT LoRA | PubMedQA | Built-in abstract (dataset) |
| 6 | **Phase 2** | Base model | MedQA | None |
| 7 | **Phase 3a** | CoT-200 LoRA | MedQA | None |
| 8 | **Phase 3b** | CoT-200 LoRA | PubMedQA | None |
| 9 | Final summary | | | |

> **Cells 1–4** are shared setup (install, imports, helpers, config & datasets).


In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================

# 1. Install Unsloth
!pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Datasets
!pip install -U datasets

# Restart the runtime after installation before running the next cells


In [ ]:
# ==========================================
# CELL 2: IMPORTS & CONFIGURATION
# ==========================================

import os
import csv
import re
import gc

import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import logging as hf_logging
from unsloth import FastLanguageModel

torch.backends.cuda.matmul.allow_tf32 = True
hf_logging.set_verbosity_error()

# ==========================================
# CONFIGURATION
# ==========================================
BASE_MODEL         = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_CONTEXT_TOKENS = 2048
MAX_NEW_TOKENS     = 512
BATCH_SIZE         = 16

# ==========================================
# SYSTEM PROMPTS
# ==========================================
SYS_MEDQA_NO_RAG = (
    "You are an expert medical AI assistant. Your task is to answer the multiple choice question. "
    "Formulate a detailed explanation using your medical knowledge, then conclude with: "
    "Conclusion: A, B, C, or D."
)

SYS_PUBMED_NO_RAG = (
    "You are a helpful and expert medical assistant. "
    "Evaluate the medical premise and conclude your reasoning with a final classification of 'yes', 'no', or 'maybe'."
)

SYS_PUBMED_BUILTIN = (
    "You are a helpful and expert medical assistant. "
    "You will be provided with a medical research abstract as context. "
    "Carefully read the context and use it to evaluate the question. "
    "Conclude your reasoning with a final classification of 'yes', 'no', or 'maybe'."
)

print("[+] Imports & config ready.")


In [ ]:
# ==========================================
# CELL 3: HELPER FUNCTIONS
# ==========================================

def extract_mcq_answer(text):
    """Extract A/B/C/D from generated text."""
    match = re.search(
        r"(?:Conclusion:|Answer:|answer is|correct (?:answer|option|choice) is)\s*([A-D])",
        text, re.IGNORECASE
    )
    if match:
        return match.group(1).upper()
    match_end = re.search(
        r"\b([A-D])\b[\.\s]*(?:<\|im_end\|>)?$",
        text, re.IGNORECASE
    )
    if match_end:
        return match_end.group(1).upper()
    return None


def extract_ynm_answer(text):
    """Extract yes/no/maybe from generated text."""
    match = re.search(
        r"(?:Conclusion:|Answer:|answer is|classification is)\s*(yes|no|maybe)",
        text, re.IGNORECASE
    )
    if match:
        return match.group(1).lower()
    match_end = re.search(
        r"\b(yes|no|maybe)\b[\.\s]*(?:<\|im_end\|>)?$",
        text, re.IGNORECASE
    )
    if match_end:
        return match_end.group(1).lower()
    return None


def build_prompt_medqa(question, options, tokenizer):
    options_text = "\n".join([f"{k}) {v}" for k, v in options.items()])
    user_content = f"Question:\n{question}\n\nOptions:\n{options_text}"
    messages = [
        {"role": "system", "content": SYS_MEDQA_NO_RAG},
        {"role": "user",   "content": user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def build_prompt_pubmedqa(question, tokenizer):
    user_content = f"Question: {question}\nAnswer:"
    messages = [
        {"role": "system", "content": SYS_PUBMED_NO_RAG},
        {"role": "user",   "content": user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(model, tokenizer, prompts):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_CONTEXT_TOKENS,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.05,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[:, input_length:]
    decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return [t.strip() for t in decoded]


def load_model(lora_path=None):
    label = f"LoRA: {lora_path}" if lora_path else "base (no LoRA)"
    print(f"\n[+] Loading model — {label}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_CONTEXT_TOKENS,
        dtype=torch.float16,
        load_in_4bit=True,
    )
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if lora_path and os.path.exists(lora_path):
        print(f"[+] Loading LoRA adapter: {lora_path}")
        model.load_adapter(lora_path)
    elif lora_path:
        print(f"[-] LoRA path not found: {lora_path}. Running base model.")
    FastLanguageModel.for_inference(model)
    model.eval()
    print("[+] Model ready.")
    return model, tokenizer


def unload_model(model, tokenizer):
    print("[+] Unloading model...")
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()


def save_results(results, output_file):
    if not results:
        print("[!] No results to save.")
        return
    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"[+] Results saved -> {output_file}")




def build_prompt_pubmedqa_builtin(question, abstract, tokenizer):
    """Build prompt for RAG-SFT PubMedQA using the dataset's own abstract."""
    # Truncate abstract to fit within context budget
    sys_tokens = len(tokenizer.encode(SYS_PUBMED_BUILTIN))
    q_tokens   = len(tokenizer.encode(question))
    budget     = MAX_CONTEXT_TOKENS - sys_tokens - q_tokens - 50
    if budget > 0:
        abs_tokens = tokenizer.encode(abstract)
        if len(abs_tokens) > budget:
            abstract = tokenizer.decode(abs_tokens[:budget], skip_special_tokens=True)
    user_content = f"Context:\n{abstract}\n\nQuestion: {question}\nAnswer:"
    messages = [
        {"role": "system", "content": SYS_PUBMED_BUILTIN},
        {"role": "user",   "content": user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("[+] Helper functions defined.")


In [ ]:
# ==========================================
# CELL 4: KAGGLE CONFIGURATION & DATASET LOAD
# ==========================================

# --- Paths (edit these) ---
# Leave LORA_PATH_COT200 empty ('') to use the pure base model
LORA_PATH_COT200 = ''   # e.g. '/kaggle/input/my-cot200-lora'
LORA_PATH_RAGSFT  = ''   # e.g. '/kaggle/input/my-ragsft-lora'

DATASET_LIMIT    = 0    # 0 = full dataset; set e.g. 50 for a quick smoke-test
OUTPUT_DIR       = '/kaggle/working'

# --- Load datasets once ---
print('[+] Loading datasets...')
medqa_ds    = load_dataset('GBaker/MedQA-USMLE-4-options', split='test')
pubmedqa_ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')

if DATASET_LIMIT > 0:
    medqa_ds    = medqa_ds.select(range(min(DATASET_LIMIT, len(medqa_ds))))
    pubmedqa_ds = pubmedqa_ds.select(range(min(DATASET_LIMIT, len(pubmedqa_ds))))

print(f'  MedQA    : {len(medqa_ds)} questions')
print(f'  PubMedQA : {len(pubmedqa_ds)} questions')

# Shared results accumulator across all phases
all_phase_summaries = []


In [ ]:
# ==========================================
# PHASE 1: PubMedQA | RAG-SFT LoRA | Built-in Abstract Context
# ==========================================
# Uses the research abstract already embedded in each PubMedQA row
# (dataset field: context -> contexts).  No external retriever needed.

PHASE_NAME  = 'Phase 1 | PubMedQA | RAG-SFT LoRA | Built-in Context'
OUTPUT_FILE = f'{OUTPUT_DIR}/phase1_ragsft_pubmedqa_builtin.csv'

model, tokenizer = load_model(lora_path=LORA_PATH_RAGSFT)

correct, total = 0, len(pubmedqa_ds)
results_p1 = []
print(f'\n[*] {PHASE_NAME}  ({total} questions)')

for start in tqdm(range(0, total, BATCH_SIZE), desc='PubMedQA-RAG-SFT', colour='green'):
    try:
        batch = pubmedqa_ds[start : start + BATCH_SIZE]
        prompts, actuals, questions = [], [], []

        for question, ctx_obj, actual in zip(batch['question'], batch['context'], batch['final_decision']):
            # Ground-truth abstract is embedded inside the dataset row
            abstract = '\n\n'.join(ctx_obj['contexts'])
            prompts.append(build_prompt_pubmedqa_builtin(question, abstract, tokenizer))
            actuals.append(actual.strip().lower())
            questions.append(question)

        outputs = generate_batch(model, tokenizer, prompts)

        for i, text in enumerate(outputs):
            pred       = extract_ynm_answer(text)
            is_correct = (pred == actuals[i])
            if is_correct: correct += 1
            results_p1.append({
                'Phase': PHASE_NAME, 'Dataset': 'PubMedQA',
                'Index': start + i + 1, 'Question': questions[i][:200],
                'Expected': actuals[i], 'Predicted': pred or '[Invalid]',
                'IsCorrect': is_correct,
            })
    except Exception as e:
        print(f'  [!] Batch {start} skipped: {e}')

acc_p1 = correct / total * 100 if total > 0 else 0
print(f'\n  -> Accuracy: {acc_p1:.2f}%  ({correct}/{total})')
all_phase_summaries.append({'Phase': PHASE_NAME, 'Correct': correct, 'Total': total, 'Accuracy': f'{acc_p1:.2f}%'})

save_results(results_p1, OUTPUT_FILE)
unload_model(model, tokenizer)

In [ ]:
# ==========================================
# PHASE 2: MedQA | Base Model | No RAG
# ==========================================

PHASE_NAME   = 'Phase 2 | MedQA | Base Model | No RAG'
OUTPUT_FILE  = f'{OUTPUT_DIR}/phase2_base_medqa_no_rag.csv'

model, tokenizer = load_model(lora_path=None)   # pure base model

correct, total = 0, len(medqa_ds)
results_p2 = []
print(f'\n[*] {PHASE_NAME}  ({total} questions)')

for start in tqdm(range(0, total, BATCH_SIZE), desc='MedQA-Base', colour='cyan'):
    try:
        batch = medqa_ds[start : start + BATCH_SIZE]
        prompts, actuals, questions = [], [], []

        for question, options, actual in zip(batch['question'], batch['options'], batch['answer_idx']):
            prompts.append(build_prompt_medqa(question, options, tokenizer))
            actuals.append(actual)
            questions.append(question)

        outputs = generate_batch(model, tokenizer, prompts)

        for i, text in enumerate(outputs):
            pred       = extract_mcq_answer(text)
            is_correct = (str(pred).upper() == str(actuals[i]).upper())
            if is_correct: correct += 1
            results_p2.append({
                'Phase': PHASE_NAME, 'Dataset': 'MedQA',
                'Index': start + i + 1, 'Question': questions[i][:200],
                'Expected': actuals[i], 'Predicted': pred or '[Invalid]',
                'IsCorrect': is_correct,
            })
    except Exception as e:
        print(f'  [!] Batch {start} skipped: {e}')

acc_p2 = correct / total * 100 if total > 0 else 0
print(f'\n  -> Accuracy: {acc_p2:.2f}%  ({correct}/{total})')
all_phase_summaries.append({'Phase': PHASE_NAME, 'Correct': correct, 'Total': total, 'Accuracy': f'{acc_p2:.2f}%'})

save_results(results_p2, OUTPUT_FILE)
unload_model(model, tokenizer)


In [ ]:
# ==========================================
# PHASE 3a: MedQA | CoT-200 LoRA | No RAG
# ==========================================

PHASE_NAME   = 'Phase 3a | MedQA | CoT-200 LoRA | No RAG'
OUTPUT_FILE  = f'{OUTPUT_DIR}/phase3a_cot200_medqa_no_rag.csv'

model, tokenizer = load_model(lora_path=LORA_PATH_COT200)

correct, total = 0, len(medqa_ds)
results_p3a = []
print(f'\n[*] {PHASE_NAME}  ({total} questions)')

for start in tqdm(range(0, total, BATCH_SIZE), desc='MedQA-CoT200', colour='cyan'):
    try:
        batch = medqa_ds[start : start + BATCH_SIZE]
        prompts, actuals, questions = [], [], []

        for question, options, actual in zip(batch['question'], batch['options'], batch['answer_idx']):
            prompts.append(build_prompt_medqa(question, options, tokenizer))
            actuals.append(actual)
            questions.append(question)

        outputs = generate_batch(model, tokenizer, prompts)

        for i, text in enumerate(outputs):
            pred       = extract_mcq_answer(text)
            is_correct = (str(pred).upper() == str(actuals[i]).upper())
            if is_correct: correct += 1
            results_p3a.append({
                'Phase': PHASE_NAME, 'Dataset': 'MedQA',
                'Index': start + i + 1, 'Question': questions[i][:200],
                'Expected': actuals[i], 'Predicted': pred or '[Invalid]',
                'IsCorrect': is_correct,
            })
    except Exception as e:
        print(f'  [!] Batch {start} skipped: {e}')

acc_p3a = correct / total * 100 if total > 0 else 0
print(f'\n  -> Accuracy: {acc_p3a:.2f}%  ({correct}/{total})')
all_phase_summaries.append({'Phase': PHASE_NAME, 'Correct': correct, 'Total': total, 'Accuracy': f'{acc_p3a:.2f}%'})

save_results(results_p3a, OUTPUT_FILE)
unload_model(model, tokenizer)


In [ ]:
# ==========================================
# PHASE 3b: PubMedQA | CoT-200 LoRA | No RAG
# ==========================================

PHASE_NAME   = 'Phase 3b | PubMedQA | CoT-200 LoRA | No RAG'
OUTPUT_FILE  = f'{OUTPUT_DIR}/phase3b_cot200_pubmedqa_no_rag.csv'

model, tokenizer = load_model(lora_path=LORA_PATH_COT200)

correct, total = 0, len(pubmedqa_ds)
results_p3b = []
print(f'\n[*] {PHASE_NAME}  ({total} questions)')

for start in tqdm(range(0, total, BATCH_SIZE), desc='PubMedQA-CoT200', colour='green'):
    try:
        batch = pubmedqa_ds[start : start + BATCH_SIZE]
        prompts, actuals, questions = [], [], []

        for question, actual in zip(batch['question'], batch['final_decision']):
            prompts.append(build_prompt_pubmedqa(question, tokenizer))
            actuals.append(actual.strip().lower())
            questions.append(question)

        outputs = generate_batch(model, tokenizer, prompts)

        for i, text in enumerate(outputs):
            pred       = extract_ynm_answer(text)
            is_correct = (pred == actuals[i])
            if is_correct: correct += 1
            results_p3b.append({
                'Phase': PHASE_NAME, 'Dataset': 'PubMedQA',
                'Index': start + i + 1, 'Question': questions[i][:200],
                'Expected': actuals[i], 'Predicted': pred or '[Invalid]',
                'IsCorrect': is_correct,
            })
    except Exception as e:
        print(f'  [!] Batch {start} skipped: {e}')

acc_p3b = correct / total * 100 if total > 0 else 0
print(f'\n  -> Accuracy: {acc_p3b:.2f}%  ({correct}/{total})')
all_phase_summaries.append({'Phase': PHASE_NAME, 'Correct': correct, 'Total': total, 'Accuracy': f'{acc_p3b:.2f}%'})

save_results(results_p3b, OUTPUT_FILE)
unload_model(model, tokenizer)


In [ ]:
# ==========================================
# FINAL SUMMARY
# ==========================================

print('\n' + '=' * 65)
print('  NON-RAG BENCHMARK — FINAL RESULTS')
print('=' * 65)
print(f"  {'Phase':<50}  {'Score':<12}  Accuracy")
print(f"  {'-'*50}  {'-'*12}  {'-'*10}")
for s in all_phase_summaries:
    score = f"{s['Correct']}/{s['Total']}"
    print(f"  {s['Phase']:<50}  {score:<12}  {s['Accuracy']}")
print('=' * 65)

# Save combined summary CSV
summary_file = f'{OUTPUT_DIR}/benchmark_summary_no_rag.csv'
with open(summary_file, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['Phase', 'Correct', 'Total', 'Accuracy'])
    writer.writeheader()
    writer.writerows(all_phase_summaries)
print(f'\n[+] Summary saved -> {summary_file}')
